# 仿真数据对比测试 —— compared1-4 模型

对 `datasets/data_simulation/` 下的 12 个 `.mat` 文件，用 4 个模型（compared1~4, epoch=200）
分别进行重建测试，对比 PSNR / SSIM 指标。

**流程**：
1. 从 GT 复振幅用 `diff_compute` 在 z=2mm 处计算衍射图
2. 衍射图归一化后输入 4 个模型，得到预测复振幅
3. 与 GT 对比计算振幅和相位的 PSNR / SSIM
4. 每文件每模型求平均，汇总输出

结果保存到 `compare_results/` 目录下（sim_ 前缀）。

In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp/PINet_cpx_cl')

import os
import numpy as np
from scipy.io import loadmat
import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.model import PINet_cpx_v6
from src.config import *
from src.utilities import (
    diff_compute,
    compute_psnr_skimage_single,
    compute_ssim_skimage_single,
)

print('Device:', DEVICE)

In [ ]:
# ==================== 全局配置 ====================

# ---- 路径 ----
SIM_DATA_DIR = os.path.join(PROJECT_ROOT, 'datasets', 'data_simulation')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'compare_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_FOLDERS = {
    'compared1': os.path.join(SAVE_DIR, 'model_saved_pinet_compared1'),
    'compared2': os.path.join(SAVE_DIR, 'model_saved_pinet_compared2'),
    'compared3': os.path.join(SAVE_DIR, 'model_saved_pinet_compared3'),
    'compared4': os.path.join(SAVE_DIR, 'model_saved_pinet_compared4'),
}
MODEL_FILENAME = 'model_pinet_size256_epoch_200_batchsize4_1.5mm_to_3.0mm.pth'

# ---- 物理参数 ----
Z_MM = 2.0
PIXEL_SIZE = 1.67e-6
Nx, Ny = 256, 256

# ---- 模型参数 ----
FOLD_ITERS = 9

# ---- 可视化参数 ----
PERCENT = 0.5

print(f'Sim data dir:  {SIM_DATA_DIR}')
print(f'Output dir:    {OUTPUT_DIR}')
print(f'Models ({len(MODEL_FOLDERS)}): {list(MODEL_FOLDERS.keys())}')
print(f'z = {Z_MM} mm, fold_iters = {FOLD_ITERS}')

In [ ]:
# ==================== 计算传递函数 (TF) —— z = 2 mm ====================
z = Z_MM * 1e-3
k = 2 * np.pi / WAVELENGTH
dx = PIXEL_SIZE

Lx = Nx * dx
Ly = Ny * dx
fx = np.linspace(-1 / (2 * dx), 1 / (2 * dx) - 1 / Lx, Nx)
fy = np.linspace(-1 / (2 * dx), 1 / (2 * dx) - 1 / Ly, Ny)
FX, FY = np.meshgrid(fy, fx)

sqrt_term = k**2 - (2 * np.pi * FX)**2 - (2 * np.pi * FY)**2
sqrt_term = np.maximum(sqrt_term, 0)  # 截断倏逝波
TF = np.exp(1j * z * np.sqrt(sqrt_term))
TF_torch = torch.from_numpy(TF).to(DEVICE).to(torch.complex64)

print(f'TF: shape={TF_torch.shape}, dtype={TF_torch.dtype}')
print(f'z = {z*1e3:.3f} mm, dx = {dx*1e6:.2f} um, Nyquist = {1/(2*dx)*1e-3:.1f} lp/mm')

In [ ]:
# ==================== 加载 4 个模型 ====================
def load_checkpoint(checkpoint_path, model):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])


models = {}
for label, folder in MODEL_FOLDERS.items():
    ckpt_path = os.path.join(folder, MODEL_FILENAME)
    print(f'[{label}] {ckpt_path}')
    print(f'         exists: {os.path.isfile(ckpt_path)}')

    m = PINet_cpx_v6(fold_iters=FOLD_ITERS).to(DEVICE)
    load_checkpoint(ckpt_path, m)
    m.eval()
    models[label] = m

model_labels = list(models.keys())
print(f'\nAll {len(models)} models loaded.')

In [ ]:
# ==================== 辅助函数 ====================
def complex_to_torch(arr, device=DEVICE):
    """将 numpy complex 数组转为 torch 复张量。"""
    real = torch.from_numpy(np.real(arr).astype(np.float32))
    imag = torch.from_numpy(np.imag(arr).astype(np.float32))
    return (real + 1j * imag).to(device)


def clip_and_normalize(img, percent=PERCENT):
    """百分位裁剪并归一化到 [0,1]。"""
    vmin = np.percentile(img, percent)
    vmax = np.percentile(img, 100 - percent)
    img = np.clip(img, vmin, vmax)
    return (img - vmin) / (vmax - vmin)

In [ ]:
# ==================== 批量测试 ====================
results = []          # 长格式记录 (文件, 模型, 指标...)
viz_data = []         # 每文件第一组 GT + 4 模型预测
mat_files = sorted([f for f in os.listdir(SIM_DATA_DIR) if f.endswith('.mat')])
print(f'共 {len(mat_files)} 个 .mat 文件\n')

for fname in mat_files:
    file_path = os.path.join(SIM_DATA_DIR, fname)
    data = loadmat(file_path)
    complex_patches = data['complex_patches']  # (N, 256, 256) complex64
    N = complex_patches.shape[0]

    short_name = fname.replace('_complex_patches_256_selected.mat', '')
    print(f'{short_name}: N={N}')

    # 按模型累计指标
    accum = {label: {'amp_psnr': [], 'amp_ssim': [], 'phs_psnr': [], 'phs_ssim': []}
             for label in model_labels}

    viz_entry = None
    for i in range(N):
        gt_complex = complex_patches[i]  # (256, 256) complex64
        gt_tensor = complex_to_torch(gt_complex)

        gt_amp = torch.abs(gt_tensor)
        gt_phs = torch.angle(gt_tensor)
        gt_amp_np = gt_amp.cpu().numpy()
        gt_phs_np = gt_phs.cpu().numpy()

        # 计算衍射图（只算一次）
        diff = diff_compute(gt_tensor, TF_torch)
        diff = (diff - diff.min()) / (diff.max() - diff.min())

        # 存储第一组可视化数据
        if i == 0:
            viz_entry = {
                'name': short_name,
                'gt_amp': gt_amp_np,
                'gt_phs': gt_phs_np,
                'preds': {},
            }

        # 逐个模型推理
        for label in model_labels:
            with torch.no_grad():
                pred, y_rec = models[label](diff.unsqueeze(0).unsqueeze(0), TF_torch)

            pred_np = pred.squeeze().detach().cpu().numpy()
            pred_amp = np.abs(pred_np)
            pred_phs = np.angle(pred_np)

            # 计算指标
            accum[label]['amp_psnr'].append(
                compute_psnr_skimage_single(torch.from_numpy(pred_amp), gt_amp.cpu()))
            accum[label]['amp_ssim'].append(
                compute_ssim_skimage_single(torch.from_numpy(pred_amp), gt_amp.cpu()))
            accum[label]['phs_psnr'].append(
                compute_psnr_skimage_single(torch.from_numpy(pred_phs), gt_phs.cpu()))
            accum[label]['phs_ssim'].append(
                compute_ssim_skimage_single(torch.from_numpy(pred_phs), gt_phs.cpu()))

            if i == 0:
                viz_entry['preds'][label] = (pred_amp, pred_phs)

    if viz_entry is not None:
        viz_data.append(viz_entry)

    # 汇总此文件的各模型指标
    for label in model_labels:
        results.append({
            'File': short_name,
            'N': N,
            'Model': label,
            'Amp_PSNR': np.mean(accum[label]['amp_psnr']),
            'Amp_PSNR_std': np.std(accum[label]['amp_psnr']),
            'Amp_SSIM': np.mean(accum[label]['amp_ssim']),
            'Amp_SSIM_std': np.std(accum[label]['amp_ssim']),
            'Phs_PSNR': np.mean(accum[label]['phs_psnr']),
            'Phs_PSNR_std': np.std(accum[label]['phs_psnr']),
            'Phs_SSIM': np.mean(accum[label]['phs_ssim']),
            'Phs_SSIM_std': np.std(accum[label]['phs_ssim']),
        })

print(f'\n===== 全部 {len(mat_files)} 个文件测试完成 =====')

In [ ]:
# ==================== 可视化：每文件 GT + 4 模型对比 ====================
for item in viz_data:
    short_name = item['name']
    n_models = len(model_labels)
    n_cols = 1 + n_models  # GT + 4 models = 5 cols

    fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 6))

    # ---- GT ----
    gt_amp_disp = clip_and_normalize(item['gt_amp'])
    gt_phs_disp = clip_and_normalize(item['gt_phs'])
    axes[0, 0].imshow(gt_amp_disp, cmap='gray')
    axes[1, 0].imshow(gt_phs_disp, cmap='gray')
    axes[0, 0].set_title('GT Amplitude')
    axes[1, 0].set_title('GT Phase')
    axes[0, 0].axis('off')
    axes[1, 0].axis('off')

    # ---- 各模型预测 ----
    for j, label in enumerate(model_labels):
        pred_amp, pred_phs = item['preds'][label]
        amp_disp = clip_and_normalize(pred_amp)
        phs_disp = clip_and_normalize(pred_phs)

        axes[0, j + 1].imshow(amp_disp, cmap='gray')
        axes[1, j + 1].imshow(phs_disp, cmap='gray')
        axes[0, j + 1].set_title(f'{label}')
        axes[1, j + 1].set_title(f'{label}')
        axes[0, j + 1].axis('off')
        axes[1, j + 1].axis('off')

    axes[0, 0].set_ylabel('Amplitude', fontsize=11)
    axes[1, 0].set_ylabel('Phase', fontsize=11)
    fig.suptitle(f'{short_name}', fontsize=13)
    plt.tight_layout()

    save_path = os.path.join(OUTPUT_DIR, f'sim_comparison_{short_name}.png')
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved: {os.path.basename(save_path)}')

In [ ]:
# ==================== 汇总结果表 ====================
df = pd.DataFrame(results)

# 透视表：行=文件, 列=(指标, 模型)
pivot_psnr = df.pivot(index='File', columns='Model', values='Amp_PSNR')
pivot_ssim = df.pivot(index='File', columns='Model', values='Amp_SSIM')
pivot_psnr_phs = df.pivot(index='File', columns='Model', values='Phs_PSNR')
pivot_ssim_phs = df.pivot(index='File', columns='Model', values='Phs_SSIM')

print('===== 振幅 PSNR (dB) =====')
print(pivot_psnr.to_string())
print(f'\n===== 振幅 SSIM =====')
print(pivot_ssim.to_string())
print(f'\n===== 相位 PSNR (dB) =====')
print(pivot_psnr_phs.to_string())
print(f'\n===== 相位 SSIM =====')
print(pivot_ssim_phs.to_string())

In [ ]:
# ==================== 各模型全局平均 ====================
print('===== 各模型全局平均（所有 12 个文件）=====')
for label in model_labels:
    sub = df[df['Model'] == label]
    print(f'\n--- {label} ---')
    print(f'  振幅 PSNR: {sub["Amp_PSNR"].mean():.2f} ± {sub["Amp_PSNR"].std():.2f} dB')
    print(f'  振幅 SSIM: {sub["Amp_SSIM"].mean():.4f} ± {sub["Amp_SSIM"].std():.4f}')
    print(f'  相位 PSNR: {sub["Phs_PSNR"].mean():.2f} ± {sub["Phs_PSNR"].std():.2f} dB')
    print(f'  相位 SSIM: {sub["Phs_SSIM"].mean():.4f} ± {sub["Phs_SSIM"].std():.4f}')

In [ ]:
# ==================== 保存指标 CSV ====================
csv_path = os.path.join(OUTPUT_DIR, 'sim_metrics_comparison.csv')
df.to_csv(csv_path, index=False)
print(f'Metrics saved to: {csv_path}')

# 同时保存各指标透视表
pivot_psnr.to_csv(os.path.join(OUTPUT_DIR, 'sim_amp_psnr_pivot.csv'))
pivot_ssim.to_csv(os.path.join(OUTPUT_DIR, 'sim_amp_ssim_pivot.csv'))
pivot_psnr_phs.to_csv(os.path.join(OUTPUT_DIR, 'sim_phs_psnr_pivot.csv'))
pivot_ssim_phs.to_csv(os.path.join(OUTPUT_DIR, 'sim_phs_ssim_pivot.csv'))
print('Pivot tables saved.')